# BÀI TẬP CHƯƠNG 2
Thao tác với bảng student và course bằng SQLite

## Khoi tao du lieu

In [ ]:
import sqlite3
import pandas as pd
from datetime import datetime, timedelta

conn = sqlite3.connect(':memory:')  
cursor = conn.cursor()

# Xóa bảng nếu tồn tại để tránh lỗi cột thừa
cursor.execute('DROP TABLE IF EXISTS student')
cursor.execute('DROP TABLE IF EXISTS course')

# Tạo bảng student
cursor.execute('''
    CREATE TABLE student (
        student_id INTEGER PRIMARY KEY,
        name Varchar(50),
        class VARCHAR(50),
        course_id INTEGER,
        score FLOAT
    )
''')

# Chèn dữ liệu vào bảng student
students = [
    (1, 'Nguyen Minh Hoang', 'May Tinh', 12, 6.7),
    (2, 'Tran Thi Lan', 'Kinh Te', 34, 9.2),
    (3, 'Pham Van Nam', 'Toan Tin', None, 7.9),
    (4, 'Le Thanh Huyen', 'Toan Tin', 20, 7.2),
    (5, 'Vu Quoc Anh', 'May Tinh', 24, 8.0),
    (6, 'Dang Thuy Linh', 'May Tinh', 24, 5.5),
    (7, 'Bui Tien Dung', 'Kinh Te', 34, 9.2),
    (8, 'Ho Ngoc Mai', 'Toan Tin', 20, 8.8),
    (9, 'Duong Huu Phuc', 'Kinh Te', None, 7.2),
    (10, 'Cao Thi Hanh', 'May Tinh', None, 7.0)
]
cursor.executemany("INSERT INTO student VALUES (?, ?, ?, ?, ?)", students)

# Tạo bảng course
cursor.execute('''
    CREATE TABLE IF NOT EXISTS course (
        id INTEGER PRIMARY KEY,
        course_name VARCHAR(50)
    )
''')

# Chèn dữ liệu vào bảng course
courses = [
    (12, 'Giai tich'),
    (34, 'Thong ke'),
    (26, 'Tin hoc')
]
cursor.executemany("INSERT INTO course VALUES (?, ?)", courses)

# Lưu thay đổi
conn.commit()
print("Dữ liệu đã được khởi tạo thành công!")
display(pd.read_sql('SELECT * FROM student', conn))

Dữ liệu đã được khởi tạo thành công!


,student_id,name,class,course_id,score
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7
1,2,Tran Thi Lan,Kinh Te,34.0,9.2
2,3,Pham Van Nam,Toan Tin,NaN,7.9
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2
4,5,Vu Quoc Anh,May Tinh,24.0,8.0
5,6,Dang Thuy Linh,May Tinh,24.0,5.5
6,7,Bui Tien Dung,Kinh Te,34.0,9.2
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2
9,10,Cao Thi Hanh,May Tinh,NaN,7.0


## Câu 1: Kết nối hai bảng

### 1.1 Tích Descartes (CROSS JOIN)

In [59]:
cursor.execute('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score,
           c.id AS course_table_id, c.course_name
    FROM student s
    CROSS JOIN course c
''')
cols = [d[0] for d in cursor.description]
df_cross = pd.DataFrame(cursor.fetchall(), columns=cols)
print(f'Tich Descartes: {len(df_cross)} dong (10 sinh vien x 3 mon hoc)')
display(df_cross)

Tich Descartes: 30 dong (10 sinh vien x 3 mon hoc)


,student_id,name,class,course_id,score,course_table_id,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,26,Tin hoc
2,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,34,Thong ke
3,2,Tran Thi Lan,Kinh Te,34.0,9.2,12,Giai tich
4,2,Tran Thi Lan,Kinh Te,34.0,9.2,26,Tin hoc
5,2,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke
6,3,Pham Van Nam,Toan Tin,NaN,7.9,12,Giai tich
7,3,Pham Van Nam,Toan Tin,NaN,7.9,26,Tin hoc
8,3,Pham Van Nam,Toan Tin,NaN,7.9,34,Thong ke
9,4,Le Thanh Huyen,Toan Tin,20.0,7.2,12,Giai tich


### 1.2 INNER JOIN

In [60]:
cursor.execute('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
    FROM student s
    INNER JOIN course c ON s.course_id = c.id
''')
cols = [d[0] for d in cursor.description]
df_inner = pd.DataFrame(cursor.fetchall(), columns=cols)
print(f'INNER JOIN: {len(df_inner)} dong - chi lay sinh vien co course_id khop voi bang course.')
display(df_inner)

INNER JOIN: 3 dong - chi lay sinh vien co course_id khop voi bang course.


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke


### 1.3 LEFT JOIN

In [ ]:
cursor.execute('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
    FROM student s
    LEFT JOIN course c ON s.course_id = c.id
''')
cols = [d[0] for d in cursor.description]
df_left = pd.DataFrame(cursor.fetchall(), columns=cols)
print(f'LEFT JOIN: {len(df_left)} dòng - giữ toàn bộ sinh viên, course_name = NULL nếu không khớp.')
display(df_left)

LEFT JOIN: 10 dong - giu toan bo sinh vien, course_name = NULL neu khong khop.


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,NaN,7.9,None
3,4,Le Thanh Huyen,Toan Tin,20.0,7.2,None
4,5,Vu Quoc Anh,May Tinh,24.0,8.0,None
5,6,Dang Thuy Linh,May Tinh,24.0,5.5,None
6,7,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
7,8,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
8,9,Duong Huu Phuc,Kinh Te,NaN,7.2,None
9,10,Cao Thi Hanh,May Tinh,NaN,7.0,None


### 1.4 RIGHT JOIN


In [ ]:
cursor.execute('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score,
           c.id AS course_table_id, c.course_name
    FROM course c
    LEFT JOIN student s ON s.course_id = c.id
''')
cols = [d[0] for d in cursor.description]
df_right = pd.DataFrame(cursor.fetchall(), columns=cols)
print(f'RIGHT JOIN: {len(df_right)} dòng - giữ toàn bộ môn học, sinh viên = NULL nếu không có ai học môn đó.')
display(df_right)

RIGHT JOIN: 4 dong - giu toan bo mon hoc, sinh vien = NULL neu khong co ai hoc mon do.


,student_id,name,class,course_id,score,course_table_id,course_name
0,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,12,Giai tich
1,NaN,None,None,NaN,NaN,26,Tin hoc
2,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,34,Thong ke
3,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,34,Thong ke


### 1.5 FULL OUTER JOIN


In [ ]:
cursor.execute('''
    SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
    FROM student s
    LEFT JOIN course c ON s.course_id = c.id

    UNION

    SELECT s.student_id, s.name, s.class, s.course_id, s.score, c.course_name
    FROM course c
    LEFT JOIN student s ON s.course_id = c.id
''')
cols = [d[0] for d in cursor.description]
df_full = pd.DataFrame(cursor.fetchall(), columns=cols)
print(f'FULL OUTER JOIN: {len(df_full)} dòng - giữ toàn bộ bản ghi từ cả hai bảng.')
display(df_full)

FULL OUTER JOIN: 11 dong - giu toan bo ban ghi tu ca hai bang.


,student_id,name,class,course_id,score,course_name
0,NaN,None,None,NaN,NaN,Tin hoc
1,1.0,Nguyen Minh Hoang,May Tinh,12.0,6.7,Giai tich
2,2.0,Tran Thi Lan,Kinh Te,34.0,9.2,Thong ke
3,3.0,Pham Van Nam,Toan Tin,NaN,7.9,None
4,4.0,Le Thanh Huyen,Toan Tin,20.0,7.2,None
5,5.0,Vu Quoc Anh,May Tinh,24.0,8.0,None
6,6.0,Dang Thuy Linh,May Tinh,24.0,5.5,None
7,7.0,Bui Tien Dung,Kinh Te,34.0,9.2,Thong ke
8,8.0,Ho Ngoc Mai,Toan Tin,20.0,8.8,None
9,9.0,Duong Huu Phuc,Kinh Te,NaN,7.2,None


---
## Câu 2: Cập nhật course_id còn thiếu và thống kê


In [64]:
# Cập nhật course_id NULL bằng giá trị trong bảng course
cursor.execute("UPDATE student SET course_id = 26 WHERE student_id = 3")   # Pham Van Nam -> Tin hoc
cursor.execute("UPDATE student SET course_id = 12 WHERE student_id = 9")   # Duong Huu Phuc -> Giai tich
cursor.execute("UPDATE student SET course_id = 26 WHERE student_id = 10")  # Cao Thi Hanh -> Tin hoc

# Xóa sinh viên có course_id không tồn tại trong bảng course
cursor.execute('''
    DELETE FROM student
    WHERE course_id NOT IN (SELECT id FROM course)
''')
conn.commit()

print('Đã cập nhật course_id và xóa bản ghi không hợp lệ.')
print('Bảng STUDENT sau khi cập nhật:')
cursor.execute('''
    SELECT s.*, c.course_name
    FROM student s
    JOIN course c ON s.course_id = c.id
''')
cols = [d[0] for d in cursor.description]
display(pd.DataFrame(cursor.fetchall(), columns=cols))

Đã cập nhật course_id và xóa bản ghi không hợp lệ.
Bảng STUDENT sau khi cập nhật:


,student_id,name,class,course_id,score,course_name
0,1,Nguyen Minh Hoang,May Tinh,12,6.7,Giai tich
1,2,Tran Thi Lan,Kinh Te,34,9.2,Thong ke
2,3,Pham Van Nam,Toan Tin,26,7.9,Tin hoc
3,7,Bui Tien Dung,Kinh Te,34,9.2,Thong ke
4,9,Duong Huu Phuc,Kinh Te,12,7.2,Giai tich
5,10,Cao Thi Hanh,May Tinh,26,7.0,Tin hoc


### 2a. Tổng số sinh viên và điểm tbinh theo từng lớp

In [65]:
cursor.execute('''
    SELECT
        class                   AS "Lop",
        COUNT(*)                AS "Tong so SV",
        ROUND(AVG(score), 2)    AS "Diem TB"
    FROM student
    GROUP BY class
    ORDER BY class
''')
cols = [d[0] for d in cursor.description]
df_2a = pd.DataFrame(cursor.fetchall(), columns=cols)
print('Tổng số sinh viên và điểm trung bình theo từng lớp:')
display(df_2a)

Tổng số sinh viên và điểm trung bình theo từng lớp:


,Lop,Tong so SV,Diem TB
0,Kinh Te,3,8.53
1,May Tinh,2,6.85
2,Toan Tin,1,7.90


### 2b. Tổng số sinh viên và điểm tbinh theo từng môn

In [66]:
cursor.execute('''
    SELECT
        c.course_name           AS "Mon hoc",
        COUNT(*)                AS "Tong so SV",
        ROUND(AVG(s.score), 2)  AS "Diem TB"
    FROM student s
    JOIN course c ON s.course_id = c.id
    GROUP BY c.id, c.course_name
    ORDER BY c.course_name
''')
cols = [d[0] for d in cursor.description]
df_2b = pd.DataFrame(cursor.fetchall(), columns=cols)
print('Tổng số sinh viên và điểm trung bình theo từng môn học:')
display(df_2b)

Tổng số sinh viên và điểm trung bình theo từng môn học:


,Mon hoc,Tong so SV,Diem TB
0,Giai tich,2,6.95
1,Thong ke,2,9.20
2,Tin hoc,2,7.45


### 2c. Phân loại theo điểm

- Diem TB >= 9.0 : Xuat sac
- 6.0 <= Diem TB <= 8.9 : Tot
- Diem TB < 6.0 : Kem

In [67]:
cursor.execute('''
    SELECT
        c.course_name           AS "Mon hoc",
        ROUND(AVG(s.score), 2)  AS "Diem TB",
        CASE
            WHEN AVG(s.score) >= 9.0 THEN 'Xuat sac'
            WHEN AVG(s.score) >= 6.0 THEN 'Tot'
            ELSE 'Kem'
        END AS "Phan loai"
    FROM student s
    JOIN course c ON s.course_id = c.id
    GROUP BY c.id, c.course_name
    ORDER BY AVG(s.score) DESC
''')
cols = [d[0] for d in cursor.description]
df_2c = pd.DataFrame(cursor.fetchall(), columns=cols)
print('Phân loại thi dựa theo điểm trung bình từng môn học:')
display(df_2c)

Phân loại thi dựa theo điểm trung bình từng môn học:


,Mon hoc,Diem TB,Phan loai
0,Thong ke,9.20,Xuat sac
1,Tin hoc,7.45,Tot
2,Giai tich,6.95,Tot


---
## Cau 3: Xếp hạng sinh viên

### 3a. Theo điểm số

In [68]:
cursor.execute('''
    SELECT
        student_id, name, class, score,
        RANK() OVER (ORDER BY score DESC) AS hang_toan_truong
    FROM student
    ORDER BY hang_toan_truong
''')
cols = [d[0] for d in cursor.description]
df_3a = pd.DataFrame(cursor.fetchall(), columns=cols)
print('Xếp hạng sinh viên theo điểm số toàn trường:')
display(df_3a)

print('Top 3 hạng cao nhất (toàn trường):')
display(df_3a[df_3a['hang_toan_truong'] <= 3])

print('Top 3 hạng thấp nhất (toàn trường):')
max_rank = df_3a['hang_toan_truong'].max()
display(df_3a[df_3a['hang_toan_truong'] > max_rank - 3].sort_values('hang_toan_truong', ascending=False))

Xếp hạng sinh viên theo điểm số toàn trường:


,student_id,name,class,score,hang_toan_truong
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Tien Dung,Kinh Te,9.2,1
2,3,Pham Van Nam,Toan Tin,7.9,3
3,9,Duong Huu Phuc,Kinh Te,7.2,4
4,10,Cao Thi Hanh,May Tinh,7.0,5
5,1,Nguyen Minh Hoang,May Tinh,6.7,6


Top 3 hạng cao nhất (toàn trường):


,student_id,name,class,score,hang_toan_truong
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Tien Dung,Kinh Te,9.2,1
2,3,Pham Van Nam,Toan Tin,7.9,3


Top 3 hạng thấp nhất (toàn trường):


,student_id,name,class,score,hang_toan_truong
5,1,Nguyen Minh Hoang,May Tinh,6.7,6
4,10,Cao Thi Hanh,May Tinh,7.0,5
3,9,Duong Huu Phuc,Kinh Te,7.2,4


### 3b. Theo từng lớp

In [69]:
cursor.execute('''
    SELECT
        student_id, name, class, score,
        RANK() OVER (PARTITION BY class ORDER BY score DESC) AS hang_trong_lop
    FROM student
    ORDER BY class, hang_trong_lop
''')
cols = [d[0] for d in cursor.description]
df_3b = pd.DataFrame(cursor.fetchall(), columns=cols)
print('Xếp hạng sinh viên theo điểm số trong từng lớp:')
display(df_3b)

print('Top 3 hạng cao nhất mỗi lớp:')
display(df_3b[df_3b['hang_trong_lop'] <= 3])

print('Top 3 hạng thấp nhất mỗi lớp:')
bottom_class = (
    df_3b.sort_values(['class', 'hang_trong_lop'], ascending=[True, False])
         .groupby('class').head(3)
         .sort_values(['class', 'hang_trong_lop'], ascending=[True, False])
)
display(bottom_class)

Xếp hạng sinh viên theo điểm số trong từng lớp:


,student_id,name,class,score,hang_trong_lop
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Tien Dung,Kinh Te,9.2,1
2,9,Duong Huu Phuc,Kinh Te,7.2,3
3,10,Cao Thi Hanh,May Tinh,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,6.7,2
5,3,Pham Van Nam,Toan Tin,7.9,1


Top 3 hạng cao nhất mỗi lớp:


,student_id,name,class,score,hang_trong_lop
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Tien Dung,Kinh Te,9.2,1
2,9,Duong Huu Phuc,Kinh Te,7.2,3
3,10,Cao Thi Hanh,May Tinh,7.0,1
4,1,Nguyen Minh Hoang,May Tinh,6.7,2
5,3,Pham Van Nam,Toan Tin,7.9,1


Top 3 hạng thấp nhất mỗi lớp:


,student_id,name,class,score,hang_trong_lop
2,9,Duong Huu Phuc,Kinh Te,7.2,3
0,2,Tran Thi Lan,Kinh Te,9.2,1
1,7,Bui Tien Dung,Kinh Te,9.2,1
4,1,Nguyen Minh Hoang,May Tinh,6.7,2
3,10,Cao Thi Hanh,May Tinh,7.0,1
5,3,Pham Van Nam,Toan Tin,7.9,1


### 3c. Theo từng môn

In [70]:
cursor.execute('''
    SELECT
        s.student_id, s.name, s.class, c.course_name, s.score,
        RANK() OVER (PARTITION BY s.course_id ORDER BY s.score DESC) AS hang_trong_mon
    FROM student s
    JOIN course c ON s.course_id = c.id
    ORDER BY c.course_name, hang_trong_mon
''')
cols = [d[0] for d in cursor.description]
df_3c = pd.DataFrame(cursor.fetchall(), columns=cols)
print('Xếp hạng sinh viên theo điểm số trong từng môn học:')
display(df_3c)

print('Top 3 hạng cao nhất mỗi môn:')
display(df_3c[df_3c['hang_trong_mon'] <= 3])

print('Top 3 hạng thấp nhất mỗi môn:')
bottom_course = (
    df_3c.sort_values(['course_name', 'hang_trong_mon'], ascending=[True, False])
         .groupby('course_name').head(3)
         .sort_values(['course_name', 'hang_trong_mon'], ascending=[True, False])
)
display(bottom_course)

Xếp hạng sinh viên theo điểm số trong từng môn học:


,student_id,name,class,course_name,score,hang_trong_mon
0,9,Duong Huu Phuc,Kinh Te,Giai tich,7.2,1
1,1,Nguyen Minh Hoang,May Tinh,Giai tich,6.7,2
2,2,Tran Thi Lan,Kinh Te,Thong ke,9.2,1
3,7,Bui Tien Dung,Kinh Te,Thong ke,9.2,1
4,3,Pham Van Nam,Toan Tin,Tin hoc,7.9,1
5,10,Cao Thi Hanh,May Tinh,Tin hoc,7.0,2


Top 3 hạng cao nhất mỗi môn:


,student_id,name,class,course_name,score,hang_trong_mon
0,9,Duong Huu Phuc,Kinh Te,Giai tich,7.2,1
1,1,Nguyen Minh Hoang,May Tinh,Giai tich,6.7,2
2,2,Tran Thi Lan,Kinh Te,Thong ke,9.2,1
3,7,Bui Tien Dung,Kinh Te,Thong ke,9.2,1
4,3,Pham Van Nam,Toan Tin,Tin hoc,7.9,1
5,10,Cao Thi Hanh,May Tinh,Tin hoc,7.0,2


Top 3 hạng thấp nhất mỗi môn:


,student_id,name,class,course_name,score,hang_trong_mon
1,1,Nguyen Minh Hoang,May Tinh,Giai tich,6.7,2
0,9,Duong Huu Phuc,Kinh Te,Giai tich,7.2,1
2,2,Tran Thi Lan,Kinh Te,Thong ke,9.2,1
3,7,Bui Tien Dung,Kinh Te,Thong ke,9.2,1
5,10,Cao Thi Hanh,May Tinh,Tin hoc,7.0,2
4,3,Pham Van Nam,Toan Tin,Tin hoc,7.9,1


---
## Câu 4: Bổ sung trườngg graduation_date
graduation_date = thoi gian hien tai + so hang tuong ung tinh theo diem so (don vi: ngay)

In [71]:
# Thêm cột graduation_date kiểu DATETIME
cursor.execute('ALTER TABLE student ADD COLUMN graduation_date DATETIME')
conn.commit()

# ấy xếp hạng toàn trường
cursor.execute('''
    SELECT student_id,
           RANK() OVER (ORDER BY score DESC) AS hang
    FROM student
''')
rank_rows = cursor.fetchall()

now = datetime.now()

# Cap nhat graduation_date cho tung sinh vien
for student_id, hang in rank_rows:
    grad_date = (now + timedelta(days=int(hang))).strftime('%Y-%m-%d %H:%M:%S')
    cursor.execute(
        'UPDATE student SET graduation_date = ? WHERE student_id = ?',
        (grad_date, student_id)
    )
conn.commit()

print(f'Thời điểm hien tai: {now.strftime("%Y-%m-%d %H:%M:%S")}')
print('Bảng STUDENT cuối cùng với graduation_date:')

cursor.execute('''
    SELECT
        s.student_id, s.name, s.class, c.course_name, s.score,
        RANK() OVER (ORDER BY s.score DESC) AS hang_toan_truong,
        s.graduation_date
    FROM student s
    JOIN course c ON s.course_id = c.id
    ORDER BY hang_toan_truong
''')
cols = [d[0] for d in cursor.description]
df_final = pd.DataFrame(cursor.fetchall(), columns=cols)
display(df_final)

Thời điểm hien tai: 2026-04-02 16:24:58
Bảng STUDENT cuối cùng với graduation_date:


,student_id,name,class,course_name,score,hang_toan_truong,graduation_date
0,2,Tran Thi Lan,Kinh Te,Thong ke,9.2,1,2026-04-03 16:24:58
1,7,Bui Tien Dung,Kinh Te,Thong ke,9.2,1,2026-04-03 16:24:58
2,3,Pham Van Nam,Toan Tin,Tin hoc,7.9,3,2026-04-05 16:24:58
3,9,Duong Huu Phuc,Kinh Te,Giai tich,7.2,4,2026-04-06 16:24:58
4,10,Cao Thi Hanh,May Tinh,Tin hoc,7.0,5,2026-04-07 16:24:58
5,1,Nguyen Minh Hoang,May Tinh,Giai tich,6.7,6,2026-04-08 16:24:58
